# Tehran Housing: A First Regression Model

Predict a home's recorded price using its area and number of rooms. Run cells from top to bottom with **Shift + Enter**.

[Open in Colab](https://colab.research.google.com/github/hghaderi-54/ai-projects/blob/main/machine-learning/notebooks/tehran-housing/Tehran_Housing_Regression.ipynb)

This educational dataset has unverified provenance and price units. The result is a historical data exercise, not a current property valuation. We retain the original `Price` units and do not label them as a currency.

Sequence: load → explore → clean → X/y → split → fit → predict → evaluate.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA_URL = "https://raw.githubusercontent.com/hghaderi-54/ai-projects/main/machine-learning/datasets/tehran-housing/tehran.csv"
candidates = [Path("../../datasets/tehran-housing/tehran.csv"), Path("machine-learning/datasets/tehran-housing/tehran.csv"), Path("tehran.csv")]
source = next((path for path in candidates if path.exists()), DATA_URL)
df = pd.read_csv(source)
display(df.head())
print("Rows and columns:", df.shape)

## Clean the input

Some areas contain commas or invalid values. Convert them to numbers, drop missing/nonpositive values, and restrict this example to homes of at most 1,000 square metres and up to 10 rooms. This is a stated scope rule, not a rule learned from the test answers. Prices are positive and remain in the original units.

We leave location out to keep the first model simple. That limits how well it can predict price.

In [ ]:
data = df[["Area", "Room", "Price"]].copy()
data["Area"] = pd.to_numeric(data["Area"].astype(str).str.replace(",", "", regex=False), errors="coerce")
for column in ["Room", "Price"]:
    data[column] = pd.to_numeric(data[column], errors="coerce")
data = data.dropna().drop_duplicates()
data = data[data["Area"].between(1, 1000) & data["Room"].between(0, 10) & (data["Price"] > 0)]
print("Usable records:", len(data))
display(data.describe())

## Explore

The plot uses a logarithmic price axis so very expensive homes do not hide the other observations. The model below still predicts prices in their original units.

In [ ]:
plt.scatter(data["Area"], data["Price"], alpha=0.35, s=12)
plt.yscale("log")
plt.xlabel("Area (square metres)")
plt.ylabel("Recorded price (original units; log scale)")
plt.title("Area and recorded price")
plt.show()

## X and y

X is a two-dimensional table: one row per home, one column per feature. y is a one-dimensional series with one price per home. We exclude `Price(USD)` from X because it is a converted version of the answer and would leak the target.

**Exercise:** choose `Area` and `Room` as X, and `Price` as y. Try it before the next cell.

In [ ]:
features = ["Area", "Room"]
X = data[features]
y = data["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

## Fit and predict

LinearRegression learns a simple relationship between features and price. A median baseline ignores the features and always predicts the training median. Compare the model with this simple alternative.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

## Evaluate

**MAE** is the average absolute error in the original price units; smaller is better. **R²** compares squared errors with predicting the test mean. It can be negative. A model that does not beat the median baseline needs improvement; a working program does not guarantee a good predictor.

In [ ]:
model_mae = mean_absolute_error(y_test, predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
print(f"Model MAE: {model_mae:,.0f}")
print(f"Median baseline MAE: {baseline_mae:,.0f}")
print(f"R-squared: {r2_score(y_test, predictions):.3f}")
print("Lower MAE than baseline:", model_mae < baseline_mae)
display(pd.DataFrame({"Actual": y_test, "Predicted": predictions}).head())

## Predict a new home

One home is still a two-dimensional table with shape `(1, 2)`. This example uses 80 square metres and 2 rooms. Linear regression can produce unrealistic or negative values, particularly outside the training range; do not use this model as a real appraisal.

**Exercise:** change the area to 100 and compare the prediction.

In [ ]:
new_home = pd.DataFrame([{"Area": 80, "Room": 2}], columns=features)
prediction = model.predict(new_home)[0]
print(f"Predicted recorded price: {prediction:,.0f} (original units)")

## What you completed

You loaded real data, applied explicit cleaning rules, separated X/y, held out test records, fit a regression model, compared it with a baseline, and predicted a new example.

Next improvements would include verified data provenance, location features, and a separate validation set. Those are outside this introductory project's scope.